In [ ]:
#Spatial coordinates with simulation points alignmend 
# File path (bct.dat_file created from the simulation files)
file_path = r"Your Path\bct.dat"

# Initialize storage for spatial points
spatial_points = []

# Read the file
with open(file_path, 'r') as file:
    lines = file.readlines()

# Read the number of spatial and temporal points from the first line
num_spatial_points, num_temporal_points = map(int, lines[0].split())

# Process the file line by line
line_idx = 1  # Start processing after the first line
for _ in range(num_spatial_points):  # Loop through each spatial point
    # Read the spatial point metadata (x, y, z, temporal points, id)
    x, y, z, temporal_points, point_id = map(float, lines[line_idx].split())
    spatial_points.append((x, y, z, int(point_id)))
    line_idx += 1  # Move to the temporal data lines
    
    # Skip the temporal data for this spatial point
    line_idx += int(temporal_points)

# Print all spatial points with their coordinates and ID
print("Spatial Points (Coordinates and ID):")
for point in spatial_points:
    x, y, z, point_id = point
    print(f"Point ID: {point_id}, Coordinates: ({x:.6f}, {y:.6f}, {z:.6f})")

# Print the total number of points
print(f"\nTotal Spatial Points Read: {len(spatial_points)}")

In [ ]:
#Compatible inflow bc file creation for simulation (header for each point:x,y,z,temporal_points, node_id, rows:dt, columns:u,v,w)
import os
import numpy as np

# File paths
bct_file_path = r"Your Path\bct.dat"
velocity_data_file = r"Your Path\Support_files\13_Cap_interpolated_timesteps_step_3.txt"
output_file_path = r"Your Path\Support_files\14_bc_interpolated.txt"

# Period and timestep size
period = 0.98  # in seconds
timesteps = 24
dt = period / (timesteps - 1)  # Adjust to ensure the last timestep equals the period

# Read bct.dat file to extract spatial point coordinates and IDs
spatial_points = []
with open(bct_file_path, 'r') as bct_file:
    lines = bct_file.readlines()
    num_spatial_points, _ = map(int, lines[0].split())
    line_idx = 1
    for _ in range(num_spatial_points):
        x, y, z, temporal_points, point_id = map(float, lines[line_idx].split())
        spatial_points.append((x, y, z, int(point_id)))
        line_idx += 1 + int(temporal_points)

# Convert spatial points to a dictionary for faster lookup
spatial_points_dict = {(x, y, z): point_id for x, y, z, point_id in spatial_points}

# Read velocity data from Coor_Cap_int_timesteps.txt
with open(velocity_data_file, 'r') as velocity_file:
    velocity_lines = velocity_file.readlines()

# Parse velocity data by timesteps
velocity_data = {}
current_timestep = None
for line in velocity_lines:
    line = line.strip()
    if line.startswith("aligned_comp_"):
        current_timestep = int(line.split('_')[-1].replace(':', ''))
        velocity_data[current_timestep] = []
    elif line and not line.startswith("x, y, z, u, v, w"):
        velocity_data[current_timestep].append(list(map(float, line.split(','))))

# Generate the output file
with open(output_file_path, 'w') as output_file:
    # Write the number of spatial points and temporal points
    output_file.write(f"{len(spatial_points)} {timesteps}\n")
    for x, y, z, point_id in spatial_points:
        # Write spatial point header
        output_file.write(f"{x:.6f} {y:.6f} {z:.6f} {timesteps} {point_id}\n")
        # Write temporal data for this point
        for timestep in range(timesteps):
            time = timestep * dt  # Adjusted to ensure the last time is exactly `period`
            # Find matching data in velocity_data
            matched_data = None
            for data in velocity_data.get(timestep, []):  # Safeguard if timestep isn't in the dictionary
                if np.isclose([x, y, z], data[:3], atol=1e-6).all():
                    matched_data = data[3:]
                    break
            if matched_data:
                u, v, w = matched_data
                output_file.write(f"{u:.6f} {v:.6f} {w:.6f} {time:.6f}\n")
            else:
                # Write zero velocities if no match is found
                output_file.write(f"0.000000 0.000000 0.000000 {time:.6f}\n")

In [ ]:
#Velocity component units conversion
input_file_path = r"Your Path\Support_files\14_bc_interpolated.txt"
output_file_path = r"Your Path\Support_files\15_bc_interpolated_cm.txt"

# Read the input file and process the velocities
with open(input_file_path, 'r') as input_file:
    lines = input_file.readlines()

# Initialize variables
output_lines = []
header_processed = False
current_point_data = []

# Process each line
for line in lines:
    line = line.strip()
    
    if not header_processed:
        # First line is the header with spatial and temporal points count
        output_lines.append(line)
        header_processed = True
        continue
    
    if len(line.split()) == 5:
        # Spatial point line: x, y, z, temporal points, id
        if current_point_data:
            # Append processed temporal data for the previous point
            output_lines.extend(current_point_data)
            current_point_data = []  # Reset for new point
        # Add the spatial point line to the output
        output_lines.append(line)
    elif len(line.split()) == 4:
        # Temporal point line: u, v, w, dt
        u, v, w, dt = map(float, line.split())
        # Convert velocities to cm/s
        u *= 100
        v *= 100
        w *= 100
        # Add the updated temporal point line to the current point data
        current_point_data.append(f"{u:.6f} {v:.6f} {w:.6f} {dt:.6f}")

# Append the last point's temporal data
if current_point_data:
    output_lines.extend(current_point_data)

# Write the updated data to the output file
with open(output_file_path, 'w') as output_file:
    output_file.write("\n".join(output_lines))

print(f"Updated file saved to: {output_file_path}")

In [ ]:
#Vector direction correction for every velocity component in each spatial and temporal point
input_file_path = r"Your Path\Support_files\15_bc_interpolated_cm.txt"
output_file_path = r"Your Path\Support_files\16_bc_interpolated_cm_corrected.txt"

def modify_and_save_velocity_data(input_path, output_path):
    with open(input_path, 'r') as file:
        lines = file.readlines()

    # Read the header
    header = lines[0].strip()
    num_points, num_timesteps = map(int, header.split())

    modified_data = [header + "\n"]  # Initialize the modified file content with the header
    i = 1  # Start after the header line

    while i < len(lines):
        # Read spatial point line (x, y, z, temporal points, id)
        spatial_point_line = lines[i].strip()
        modified_data.append(spatial_point_line + "\n")
        i += 1

        # Modify velocity data for each timestep
        for _ in range(num_timesteps):
            velocity_line = lines[i].strip().split()
            u, v, w, t = map(float, velocity_line)

            # Perform the transformation
            new_u = w  # Set u as w 
            new_v = u  # Set v as u
            new_w = -v  # Set w as -v

            # Write modified velocities to the output
            modified_data.append(f"{new_u:.6f} {new_v:.6f} {new_w:.6f} {t:.6f}\n")
            i += 1

    # Save the modified data to the output file
    with open(output_path, 'w') as output_file:
        output_file.writelines(modified_data)

    print(f"Modified data saved to: {output_path}")

# Call the function to process and save the file
modify_and_save_velocity_data(input_file_path, output_file_path)